## 01_Raw_Ingestion

In [0]:
import requests
import json
from datetime import datetime, timedelta, timezone

In [0]:
BASE_URL = "https://hapi.fhir.org/baseR4"

RESOURCE_TYPES = [
    "Patient",
    "Encounter",
    "Observation",
    "Condition"
]

PAGE_SIZE = 100

print("Base URL", BASE_URL)
print("Resources", RESOURCE_TYPES)
print("page size", PAGE_SIZE)

In [0]:
# FOR Inceremental Date Range

END_DATE = datetime.now(timezone.utc).date()
START_DATE = END_DATE - timedelta(days = 3)

print("Start Date", START_DATE)
print("End Date", END_DATE)

In [0]:

#Test the API

url = f"{BASE_URL}/Patient"

params = {
 "_lastUpdated" : f"ge{START_DATE.isoformat()}",
 "_lastUpdated" : f"lt{END_DATE.isoformat()}",
 "_count" : PAGE_SIZE
}

response = requests.get(url, params= params, timeout=60)
print("HTTP Status", response.status_code)
print("Final Status", response.url)


In [0]:
#Convert API response to JSON

if response.status_code == 200:
    bundle = response.json()

    print("Resource Type:", bundle.get("resourceType"))
    print("Bundle Type:", bundle.get("type"))
    print("Total:", bundle.get("total"))
    print("Entries Returned:", len(bundle.get("entry", [])))
else:
    print("API Error:", response.text)

In [0]:

# Inspect First Patient


entries = bundle.get("entry", [])

if entries:
    first_resource = entries[0].get("resource", {})

    print(json.dumps(first_resource, indent=2))
else:
    print("No records found.")

In [0]:
def fetch_fhir_resource(resource_type, start_date, end_date, page_size=100):

    # Keep FHIR resource name with correct capitalization
    resource_type = resource_type.capitalize()

    url = f"{BASE_URL}/{resource_type}"

    params = [
        ("_lastUpdated", f"ge{start_date.isoformat()}"),
        ("_lastUpdated", f"lt{end_date.isoformat()}"),
        ("_count", page_size)
    ]

    all_entries = []
    page_number = 1
    api_calls = []

    while url:

        print(f"Fetching {resource_type} - Page {page_number}")
        print(f"URL: {url}")

        response = requests.get(
            url,
            params=params if page_number == 1 else None,
            timeout=60
        )

        print("HTTP Status:", response.status_code)

        response.raise_for_status()

        bundle = response.json()

        extraction_timestamp = datetime.now(timezone.utc).isoformat()

        api_calls.append({
            "resource_type": resource_type,
            "page_number": page_number,
            "api_url_or_params": response.url,
            "extraction_timestamp": extraction_timestamp,
            "http_status": response.status_code
        })

        entries = bundle.get("entry", [])

        all_entries.extend(entries)

        print(f"Records returned: {len(entries)}")

        # Find next page
        next_url = None

        for link in bundle.get("link", []):

            if link.get("relation") == "next":
                next_url = link.get("url")
                break

        url = next_url
        page_number += 1

    return all_entries, api_calls

In [0]:
# ==============================
# Test Patient Pagination
# ==============================

patient_entries, patient_api_logs = fetch_fhir_resource(
    resource_type="Patient",
    start_date=START_DATE,
    end_date=END_DATE,
    page_size=PAGE_SIZE
)

print("Total Patient Entries:", len(patient_entries))
print("Total API Calls:", len(patient_api_logs))

In [0]:
# ==============================
# Extract All FHIR Resources
# ==============================

all_resource_data = {}

for resource_type in RESOURCE_TYPES:

    entries, api_logs = fetch_fhir_resource(
        resource_type=resource_type,
        start_date=START_DATE,
        end_date=END_DATE,
        page_size=PAGE_SIZE
    )

    all_resource_data[resource_type] = {
        "entries": entries,
        "api_logs": api_logs
    }

    print(
        f"{resource_type}: "
        f"{len(entries)} records, "
        f"{len(api_logs)} API calls"
    )

In [0]:
RAW_BASE_PATH = "/Volumes/workspace/fhir/raw_data"

print(RAW_BASE_PATH)

In [0]:
page_number = 1

In [0]:
def fetch_and_save_fhir_resource(
    resource_type,
    start_date,
    end_date,
    page_size=100
):

    # Make sure resource name has correct capitalization
    resource_type = resource_type.capitalize()

    url = f"{BASE_URL}/{resource_type}"

    params = [
        ("_lastUpdated", f"ge{start_date.isoformat()}"),
        ("_lastUpdated", f"lt{end_date.isoformat()}"),
        ("_count", page_size)
    ]

    # Initialize variables
    page_number = 1
    total_records = 0
    api_logs = []

    extraction_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

    while url:

        print(f"Fetching {resource_type} - Page {page_number}")

        response = requests.get(
            url,
            params=params if page_number == 1 else None,
            timeout=60
        )

        print("HTTP Status:", response.status_code)

        response.raise_for_status()

        bundle = response.json()

        # Timestamp for this API call
        extraction_timestamp = datetime.now(timezone.utc).isoformat()

        # Actual URL used
        api_url_or_params = response.url

        # Extract records from Bundle
        entries = bundle.get("entry", [])

        total_records += len(entries)

        # Raw file path
        raw_path = (
            f"{RAW_BASE_PATH}/"
            f"{resource_type}/"
            f"extraction_date={extraction_date}/"
            f"page_{page_number:03d}.json"
        )

        # Save complete FHIR Bundle as-is
        dbutils.fs.put(
            raw_path,
            json.dumps(bundle),
            overwrite=True
        )

        # Store API audit information
        api_logs.append({
            "resource_type": resource_type,
            "page_number": page_number,
            "extraction_timestamp": extraction_timestamp,
            "api_url_or_params": api_url_or_params,
            "http_status": response.status_code,
            "record_count": len(entries),
            "raw_path": raw_path
        })

        print(
            f"Saved {len(entries)} records → {raw_path}"
        )

        # Find next page
        next_url = None

        for link in bundle.get("link", []):

            if link.get("relation") == "next":
                next_url = link.get("url")
                break

        # Move to next page
        url = next_url
        page_number += 1

    print(
        f"Completed {resource_type}: "
        f"{total_records} records"
    )

    return api_logs

In [0]:
raw_api_logs = []

for resource_type in RESOURCE_TYPES:

    logs = fetch_and_save_fhir_resource(
        resource_type=resource_type,
        start_date=START_DATE,
        end_date=END_DATE,
        page_size=PAGE_SIZE
    )

    raw_api_logs.extend(logs)

    print(
        f"{resource_type} completed - "
        f"{len(logs)} API calls"
    )

print("================================")
print("RAW INGESTION COMPLETED")
print("Total API calls:", len(raw_api_logs))
print("================================")

In [0]:
api_log_df = spark.createDataFrame(raw_api_logs)

display(api_log_df)

In [0]:
from pyspark.sql.functions import current_timestamp

api_log_df = api_log_df.withColumn(
    "raw_save_timestamp",
    current_timestamp()
)

display(api_log_df)

In [0]:
metadata_path = f"{RAW_BASE_PATH}/_metadata"

dbutils.fs.mkdirs(metadata_path)

print("Metadata folder created:")
print(metadata_path)

In [0]:
api_log_path = f"{RAW_BASE_PATH}/_metadata/api_calls"

(
    api_log_df
    .write
    .format("delta")
    .mode("append")
    .save(api_log_path)
)

print("API audit metadata saved successfully.")

In [0]:
display(
    spark.read
    .format("delta")
    .load(api_log_path)
)

In [0]:
from datetime import datetime, timezone

extraction_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

print("Extraction date:", extraction_date)

In [0]:
display(
    dbutils.fs.ls(RAW_BASE_PATH)
)

In [0]:
display(
    dbutils.fs.ls(
        f"{RAW_BASE_PATH}/Patient/extraction_date={extraction_date}"
    )
)

In [0]:
sample_path = (
    f"{RAW_BASE_PATH}/Patient/"
    f"extraction_date={extraction_date}/"
    f"page_001.json"
)

sample_raw = dbutils.fs.head(sample_path, 5000)

print(sample_raw)

In [0]:
audit_df = (
    spark.read
    .format("delta")
    .load(api_log_path)
)

display(
    audit_df
    .groupBy("resource_type")
    .agg(
        {"record_count": "sum"}
    )
)

In [0]:
display(
    dbutils.fs.ls(
        f"{RAW_BASE_PATH}/Patient/"
    )
)